In [ ]:
import torch.nn as nn
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch.optim as optim
from sklearn.model_selection import train_test_split
import sympy.printing

PIXSHAPE = 96
INPUTSIZE = PIXSHAPE * PIXSHAPE
TESTSIZE = 3524

df = pd.read_csv('data/training.csv')
left_eye_center_x_df = df.dropna(axis=0, subset='left_eye_center_x')

n_records = left_eye_center_x_df.shape[0]

TEST_SIZE = n_records // 5
TRAIN_SIZE = n_records - TEST_SIZE
train_df, test_df = train_test_split(left_eye_center_x_df, test_size=0.2, random_state=42)
print(train_df.shape)

(5631, 31)


In [2]:
#cosine similarity of facial embeddings
from numpy import dot, sqrt
 
def cosine_similarity(x, y):
    return dot(x, y) / (sqrt(dot(x, x)) * sqrt(dot(y, y)))

In [17]:
class NN(nn.Module):
    def __init__(self):
        super(NN, self).__init__()
        layers = []
        layers.append(nn.Conv2d(1, 32, 3, padding=1))
        layers.append(nn.ReLU())
        layers.append(nn.MaxPool2d(2))
        layers.append(nn.Flatten())
        layers.append(nn.Linear(32*48*48, 128))
        layers.append(nn.ReLU())
        layers.append(nn.Linear(128,1))
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

In [18]:
def train_model(images: pd.Series, expected: pd.Series, neural_net: nn.Module, loss_function, optimiser, epochs=1000):
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max = 1000)
    x_train = torch.tensor(images.to_list(), dtype=torch.float32).view(-1,1,96,96)
    y_train = torch.tensor(expected.to_list()).unsqueeze(1)
    for epoch in range(0, epochs):
        optimiser.zero_grad() #clearing out the previous optimiser data
        print(f"Running epoch {epoch}")
        y_predict = neural_net(x_train)
        compute_loss = loss_function(y_predict, y_train)
        print(f"Initial loss:{compute_loss.item()}")
        compute_loss.backward()
        optimiser.step()
        scheduler.step()

   

In [19]:
def test_model(test_images, labels, neural_net: nn.Module, loss_function):
    test_x = torch.tensor(test_images.to_list(), dtype=torch.float32).view(-1,1,96,96)
    test_y = torch.tensor(labels.to_list()).unsqueeze(1)
    neural_net.eval()
    with torch.no_grad():
        loss_val = loss_function(neural_net(test_x), test_y)
    neural_net.train()
    return loss_val

In [20]:
def transform_image(img_str: list[str]) -> list[float]:
    return [float(pixel_value) / 255.0 for pixel_value in img_str.split()]


neural_net = NN()
inputs = train_df.loc[:, 'Image'].map(transform_image)
expected = train_df.loc[:, "left_eye_center_x"].map(lambda left_eye_center_x: left_eye_center_x/96.0)
with torch.no_grad():
    neural_net.model[-1].bias.fill_(float(expected.mean()))
optimiser = optim.Adam(neural_net.parameters(), lr=1e-3)
loss_function = nn.MSELoss()
test_inputs = test_df.loc[:, 'Image'].map(transform_image)
test_labels = test_df.loc[:, 'left_eye_center_x'].map(lambda left_eye_center_x: left_eye_center_x/96.0)


train_model(images=inputs, expected=expected, neural_net=neural_net, loss_function=loss_function, optimiser=optimiser)
test_model(test_images=test_inputs, labels=test_labels, neural_net=neural_net, loss_function=loss_function)


AttributeError: module 'sympy' has no attribute 'printing'